In [1]:
import pandas as pd
import numpy as np

# Load all datasets
spotify = pd.read_csv('../data/raw/spotify_real/278k_song_labelled.csv')
temperature = pd.read_csv('../data/raw/weather_real/temperature.csv')
humidity = pd.read_csv('../data/raw/weather_real/humidity.csv')
weather_desc = pd.read_csv('../data/raw/weather_real/weather_description.csv')

# Parse datetime
temperature['datetime'] = pd.to_datetime(temperature['datetime'])
humidity['datetime'] = pd.to_datetime(humidity['datetime'])
weather_desc['datetime'] = pd.to_datetime(weather_desc['datetime'])

print("All datasets loaded successfully")
print("Date range:", temperature['datetime'].min(), "to", temperature['datetime'].max())

All datasets loaded successfully
Date range: 2012-10-01 12:00:00 to 2017-11-30 00:00:00


In [2]:
# Pick few reps US cities to keep things focused
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Seattle']

# Melt temperature into long format
temp_long = temperature[['datetime'] + cities].melt(
    id_vars='datetime', var_name='city', value_name='temperature_k'
)

# Melt humidity into long format
hum_long = humidity[['datetime'] + cities].melt(
    id_vars='datetime', var_name='city', value_name='humidity'
)

# Melt weather description into long format
desc_long = weather_desc[['datetime'] + cities].melt(
    id_vars='datetime', var_name='city', value_name='weather_desc'
)

# Merge all three on datetime + city
weather_long = temp_long.merge(hum_long, on=['datetime', 'city'])
weather_long = weather_long.merge(desc_long, on=['datetime', 'city'])

# Convert temperature from Kelvin to Celsius
weather_long['temperature_c'] = weather_long['temperature_k'] - 273.15

# Drop rows with missing values
weather_long = weather_long.dropna()

print("Weather long shape:", weather_long.shape)
print(weather_long.head())

Weather long shape: (222962, 6)
             datetime      city  temperature_k  humidity weather_desc  \
1 2012-10-01 13:00:00  New York     288.220000      58.0   few clouds   
2 2012-10-01 14:00:00  New York     288.247676      57.0   few clouds   
3 2012-10-01 15:00:00  New York     288.326940      57.0   few clouds   
4 2012-10-01 16:00:00  New York     288.406203      57.0   few clouds   
5 2012-10-01 17:00:00  New York     288.485467      57.0   few clouds   

   temperature_c  
1      15.070000  
2      15.097676  
3      15.176940  
4      15.256203  
5      15.335467  


In [3]:
# Map detailed weather descriptions to simplified categories
def simplify_weather(desc):
    desc = str(desc).lower()
    if 'rain' in desc or 'drizzle' in desc or 'shower' in desc:
        return 'rainy'
    elif 'snow' in desc or 'sleet' in desc or 'blizzard' in desc:
        return 'snowy'
    elif 'clear' in desc or 'sunny' in desc:
        return 'clear'
    elif 'cloud' in desc or 'overcast' in desc:
        return 'cloudy'
    elif 'fog' in desc or 'mist' in desc or 'haze' in desc:
        return 'foggy'
    elif 'storm' in desc or 'thunder' in desc:
        return 'stormy'
    else:
        return 'other'

weather_long['weather_category'] = weather_long['weather_desc'].apply(simplify_weather)

# Aggregate to daily level
weather_long['date'] = weather_long['datetime'].dt.date
daily_weather = weather_long.groupby(['date', 'city', 'weather_category']).agg(
    avg_temp_c=('temperature_c', 'mean'),
    avg_humidity=('humidity', 'mean')
).reset_index()

print("Daily weather shape:", daily_weather.shape)
print("\nWeather category distribution:")
print(daily_weather['weather_category'].value_counts())
print("\nSample:")
print(daily_weather.head())

Daily weather shape: (25291, 5)

Weather category distribution:
weather_category
cloudy    7720
clear     6720
rainy     4859
foggy     4836
stormy     568
snowy      471
other      117
Name: count, dtype: int64

Sample:
         date         city weather_category  avg_temp_c  avg_humidity
0  2012-10-01      Chicago           cloudy   11.402669     68.909091
1  2012-10-01      Houston            clear   15.168258     91.000000
2  2012-10-01      Houston           cloudy   15.368182     84.428571
3  2012-10-01  Los Angeles            clear   18.694151     88.000000
4  2012-10-01  Los Angeles            foggy   18.720000     88.000000


In [4]:
# Compute average audio features per mood label from Spotify
mood_profiles = spotify.groupby('labels').agg(
    avg_valence=('valence', 'mean'),
    avg_energy=('energy', 'mean'),
    avg_danceability=('danceability', 'mean'),
    avg_tempo=('tempo', 'mean'),
    avg_acousticness=('acousticness', 'mean'),
    count=('valence', 'count')
).reset_index()

# Map numeric labels to mood names based on valence ordering we found
label_map = {1: 'happy', 2: 'neutral', 0: 'sad', 3: 'very_sad'}
mood_profiles['mood'] = mood_profiles['labels'].map(label_map)

print("Mood profiles:")
print(mood_profiles.round(3))

# Now aggregate daily weather stats per weather category
weather_agg = daily_weather.groupby('weather_category').agg(
    avg_temp=('avg_temp_c', 'mean'),
    avg_humidity=('avg_humidity', 'mean'),
    day_count=('date', 'count')
).reset_index()

print("\nWeather aggregated stats:")
print(weather_agg.round(3))

Mood profiles:
   labels  avg_valence  avg_energy  avg_danceability  avg_tempo  \
0       0        0.375       0.397             0.505    114.803   
1       1        0.602       0.691             0.678    121.186   
2       2        0.445       0.870             0.499    134.000   
3       3        0.217       0.183             0.391    106.265   

   avg_acousticness   count      mood  
0             0.584   82058       sad  
1             0.210  106429     happy  
2             0.033   47065   neutral  
3             0.840   42386  very_sad  

Weather aggregated stats:
  weather_category  avg_temp  avg_humidity  day_count
0            clear    15.619        65.361       6720
1           cloudy    14.990        68.838       7720
2            foggy    14.400        81.347       4836
3            other    21.890        53.282        117
4            rainy    15.108        77.690       4859
5            snowy    -1.310        77.742        471
6           stormy    24.048        71.969  

In [6]:
# A combined dataset by assigning mood distributions to weather conditions
# rainy/cloudy -> lower valence, clear -> higher valence
np.random.seed(42)

# For each weather category, sample songs proportionally
# (simulating that weather influences what mood of music people play)
weather_mood_map = {
    'clear':  [0.10, 0.50, 0.25, 0.15],  # mostly happy
    'cloudy': [0.25, 0.30, 0.25, 0.20],  # mixed
    'rainy':  [0.30, 0.15, 0.25, 0.30],  # sad leaning
    'foggy':  [0.25, 0.20, 0.25, 0.30],  # sad leaning
    'snowy':  [0.20, 0.25, 0.30, 0.25],  # mixed/sad
    'stormy': [0.15, 0.20, 0.30, 0.35],  # very sad leaning
    'other':  [0.25, 0.25, 0.25, 0.25],  # uniform
}
# order: sad(0), happy(1), neutral(2), very_sad(3)

rows = []
for _, weather_row in daily_weather.iterrows():
    cat = weather_row['weather_category']
    probs = weather_mood_map[cat]
    sampled_label = np.random.choice([0, 1, 2, 3], p=probs)
    mood_row = mood_profiles[mood_profiles['labels'] == sampled_label].iloc[0]
    rows.append({
        'date': weather_row['date'],
        'city': weather_row['city'],
        'weather_category': cat,
        'avg_temp_c': weather_row['avg_temp_c'],
        'avg_humidity': weather_row['avg_humidity'],
        'mood_label': sampled_label,
        'mood': mood_row['mood'],
        'valence': mood_row['avg_valence'],
        'energy': mood_row['avg_energy'],
        'danceability': mood_row['avg_danceability'],
        'tempo': mood_row['avg_tempo'],
        'acousticness': mood_row['avg_acousticness'],
    })

merged_df = pd.DataFrame(rows)
print("Merged dataset shape:", merged_df.shape)
print("\nSample:")
print(merged_df.head())
print("\nValence by weather category:")
print(merged_df.groupby('weather_category')['valence'].mean().round(3).sort_values(ascending=False))

Merged dataset shape: (25291, 12)

Sample:
         date         city weather_category  avg_temp_c  avg_humidity  \
0  2012-10-01      Chicago           cloudy   11.402669     68.909091   
1  2012-10-01      Houston            clear   15.168258     91.000000   
2  2012-10-01      Houston           cloudy   15.368182     84.428571   
3  2012-10-01  Los Angeles            clear   18.694151     88.000000   
4  2012-10-01  Los Angeles            foggy   18.720000     88.000000   

   mood_label      mood   valence    energy  danceability       tempo  \
0           1     happy  0.602190  0.690979      0.677576  121.186304   
1           3  very_sad  0.216824  0.182639      0.390750  106.265404   
2           2   neutral  0.444663  0.869810      0.499236  134.000134   
3           1     happy  0.602190  0.690979      0.677576  121.186304   
4           0       sad  0.374767  0.396732      0.504659  114.802695   

   acousticness  
0      0.210132  
1      0.839515  
2      0.032767  
3      

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# One-hot encode weather category
merged_encoded = pd.get_dummies(merged_df, columns=['weather_category'], drop_first=False)

# Features and target
feature_cols = ['avg_temp_c', 'avg_humidity'] + [c for c in merged_encoded.columns if c.startswith('weather_category_')]
X = merged_encoded[feature_cols]
y = merged_encoded['valence']

# Scale numerical features
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[['avg_temp_c', 'avg_humidity']] = scaler.fit_transform(X[['avg_temp_c', 'avg_humidity']])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nFeatures used:", feature_cols)

# Save processed data
merged_df.to_csv('../data/processed/merged_weather_mood.csv', index=False)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\nAll processed files saved!")